In [1]:
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import plotly.express as px
from dash import no_update
import datetime as dt

# 创建 Dash 应用
app = dash.Dash(__name__)
# 允许"回调还没执行时输出容器是空的"，不报异常
app.config.suppress_callback_exceptions = True

# 读取山火数据（本地文件）
df = pd.read_csv('Datasets/Historical_Wildfires.csv')
# 从 Date 里提取"月份名"（January...）和"年份"
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

print('数据准备完成，年份范围：', df['Year'].min(), '-', df['Year'].max())
print('区域：', sorted(df['Region'].unique().tolist()))

数据准备完成，年份范围： 2005 - 2020
区域： ['NSW', 'NT', 'QL', 'SA', 'TA', 'VI', 'WA']


In [2]:
app.layout = html.Div(children=[
    # 标题
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    html.Div([                       # 外层容器
        html.Div([                   # 左：输入区
            html.H2('Select Region:', style={'margin-right': '2em'}),
            # 单选框：7 个区域，默认选 NSW，inline=True 让选项横向排列
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           value='NSW', id='region', inline=True),
            # 下拉框：年份（从数据里取出去重后的年份），默认 2005
            html.Div([
                html.H2('Select Year:', style={'margin-right': '2em'}),
                dcc.Dropdown(df.Year.unique(), value=2005, id='year')
            ]),
        ]),
        html.Div([                   # 右：两个空的输出容器
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])

print('布局创建完成！')

布局创建完成！


In [3]:
@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value')])
def reg_year_display(input_region, input_year):
    # 先按"区域 + 年份"筛选数据
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    # 图 1：按月求"平均过火面积" → 饼图
    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region, input_year))

    # 图 2：按月求"平均火点像素数" → 柱状图
    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region, input_year))

    # 把两张图（用 dcc.Graph 包起来）返回给 plot1 / plot2
    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]

In [4]:
try:
    app.run(jupyter_mode='inline')
    print('Dash 应用已启动（notebook 内联模式）。')
except Exception as e:
    print('内联启动受限（不影响代码本身正确性）：', e)

Dash 应用已启动（notebook 内联模式）。


In [5]:
def _collect_figures(node):
    """从 Dash 组件树里把所有图（dcc.Graph 的 figure）收集出来。
    仅用于在 notebook 里"预览"回调结果。"""
    figs = []
    # 情况一：本身就是一个列表/元组（回调常返回列表），逐个往里找
    if isinstance(node, (list, tuple)):
        for item in node:
            figs += _collect_figures(item)
        return figs
    # 情况二：本身是 dcc.Graph，直接取它的 figure
    if getattr(node, 'figure', None) is not None:
        figs.append(node.figure)
    # 情况三：是 html.Div 之类的容器，往下找它的 children
    ch = getattr(node, 'children', None)
    if ch is not None:
        if not isinstance(ch, (list, tuple)):
            ch = [ch]
        for c in ch:
            figs += _collect_figures(c)
    return figs

# 调用回调看看新南威尔士（NSW）在 2005 年的两张图
graphs = reg_year_display('NSW', 2005)
print('回调返回了', len(graphs), '个图：')
for i, f in enumerate(_collect_figures(graphs), 1):
    print('--- 图', i, '---')
    f.show()

回调返回了 2 个图：
--- 图 1 ---


--- 图 2 ---


In [6]:
# 再换个区域和年份试试（昆士兰，2019）
graphs2 = reg_year_display('QL', 2019)
for i, f in enumerate(_collect_figures(graphs2), 1):
    print('--- 图', i, '---')
    f.show()

--- 图 1 ---


--- 图 2 ---
